In [4]:

import polars as pl

NULLS = ["NA"]

weather_schema = {
    "temp": pl.Float64,
    "dewp": pl.Float64,
    "humid": pl.Float64,
    "wind_dir": pl.Float64,
    "wind_speed": pl.Float64,
    "wind_gust": pl.Float64,
    "precip": pl.Float64,
    "pressure": pl.Float64,
    "visib": pl.Float64,
}
flights_schema = {
    "dep_delay": pl.Float64,
    "arr_delay": pl.Float64,
    "air_time":  pl.Float64,
    "distance":  pl.Float64,
}

airlines = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airlines.csv",
    null_values=NULLS,
    infer_schema_length=10000,
)

airports = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airports.csv",
    null_values=NULLS,
    infer_schema_length=10000,
)

flights = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_flights.csv",
    null_values=NULLS,
    infer_schema_length=10000,
    schema_overrides=flights_schema,
)

planes = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_planes.csv",
    null_values=NULLS,
    infer_schema_length=10000,
)

weather = pl.read_csv(
    "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_weather.csv",
    null_values=NULLS,
    infer_schema_length=10000,
    schema_overrides=weather_schema,
)


flights = flights.with_columns(pl.col("time_hour").str.strptime(pl.Datetime, strict=False))
weather = weather.with_columns(pl.col("time_hour").str.strptime(pl.Datetime, strict=False))


ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True,
)

print("Setup complete! Registered tables:")
print(ctx.execute("SHOW TABLES"))



Setup complete! Registered tables:
shape: (5, 1)
┌──────────┐
│ name     │
│ ---      │
│ str      │
╞══════════╡
│ airlines │
│ airports │
│ flights  │
│ planes   │
│ weather  │
└──────────┘


/tmp/ipython-input-459336499.py:60: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


In [5]:
#question1
result = ctx.execute("""
SELECT DISTINCT carrier
FROM airlines
ORDER BY carrier
""")
result

carrier
str
"""9E"""
"""AA"""
"""AS"""
"""B6"""
"""DL"""
…
"""UA"""
"""US"""
"""VX"""


In [6]:
#question1.1
result = ctx.execute("""
SELECT DISTINCT carrier
FROM airlines
ORDER BY carrier
""")
result

carrier
str
"""9E"""
"""AA"""
"""AS"""
"""B6"""
"""DL"""
…
"""UA"""
"""US"""
"""VX"""


In [7]:
#question1.2
result = ctx.execute("""
SELECT dest, COUNT(*) AS flights_cnt
FROM flights
GROUP BY dest
ORDER BY flights_cnt DESC
LIMIT 10
""")
result

dest,flights_cnt
str,u32
"""ORD""",17283
"""ATL""",17215
"""LAX""",16174
"""BOS""",15508
"""MCO""",14082
"""CLT""",14064
"""SFO""",13331
"""FLL""",12055
"""MIA""",11728


In [8]:
#question1.3
result = ctx.execute("""
SELECT *
FROM flights
WHERE dep_delay > 120
""")
result

year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
i64,i64,i64,i64,i64,f64,i64,i64,f64,str,i64,str,str,str,f64,f64,i64,i64,"datetime[μs, UTC]"
2013,1,1,848,1835,853.0,1001,1950,851.0,"""MQ""",3944,"""N942MQ""","""JFK""","""BWI""",41.0,184.0,18,35,2013-01-01 23:00:00 UTC
2013,1,1,957,733,144.0,1056,853,123.0,"""UA""",856,"""N534UA""","""EWR""","""BOS""",37.0,200.0,7,33,2013-01-01 12:00:00 UTC
2013,1,1,1114,900,134.0,1447,1222,145.0,"""UA""",1086,"""N76502""","""LGA""","""IAH""",248.0,1416.0,9,0,2013-01-01 14:00:00 UTC
2013,1,1,1540,1338,122.0,2020,1825,115.0,"""B6""",705,"""N570JB""","""JFK""","""SJU""",193.0,1598.0,13,38,2013-01-01 18:00:00 UTC
2013,1,1,1815,1325,290.0,2120,1542,338.0,"""EV""",4417,"""N17185""","""EWR""","""OMA""",213.0,1134.0,13,25,2013-01-01 18:00:00 UTC
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2013,9,30,1823,1545,158.0,1934,1733,121.0,"""9E""",3459,"""N916XJ""","""JFK""","""BNA""",95.0,765.0,15,45,2013-09-30 19:00:00 UTC
2013,9,30,1951,1649,182.0,2157,1903,174.0,"""EV""",4294,"""N13988""","""EWR""","""SAV""",95.0,708.0,16,49,2013-09-30 20:00:00 UTC
2013,9,30,2053,1815,158.0,2310,2054,136.0,"""EV""",5292,"""N600QX""","""EWR""","""ATL""",91.0,746.0,18,15,2013-09-30 22:00:00 UTC


In [9]:
#question2
#question2.1
result = ctx.execute("""
SELECT
  origin,
  AVG(dep_delay) AS avg_dep_delay
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY origin
ORDER BY avg_dep_delay DESC
""")
result

origin,avg_dep_delay
str,f64
"""EWR""",15.107954
"""JFK""",12.112159
"""LGA""",10.346876


In [10]:
#question2.2
_ = ctx.execute("""
    SELECT *
    FROM flights
    LIMIT 5
""")

result = ctx.execute("""
SELECT
  month,
  COUNT(*) AS flights_cnt
FROM flights
GROUP BY month
ORDER BY flights_cnt DESC
""")
result

month,flights_cnt
i64,u32
7,29425
8,29327
10,28889
3,28834
5,28796
…,…
12,28135
9,27574
11,27268


In [11]:
#question2.3
result = ctx.execute("""
SELECT
  carrier,
  AVG(CASE WHEN dep_delay <= 15 THEN 1 ELSE 0 END) AS on_time_rate
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY carrier
ORDER BY on_time_rate DESC
""")
result

carrier,on_time_rate
str,f64
"""HA""",0.929825
"""US""",0.878227
"""AS""",0.867978
"""AA""",0.840713
"""DL""",0.836812
…,…
"""FL""",0.733291
"""WN""",0.731027
"""F9""",0.718475


In [12]:
#question3
#question3.1
result = ctx.execute("""
SELECT
  f.carrier,
  a.name AS airline_name,
  f.flight,
  f.origin,
  f.dest
FROM flights AS f
JOIN airlines AS a
  ON f.carrier = a.carrier
LIMIT 20
""")
result

carrier,airline_name,flight,origin,dest
str,str,i64,str,str
"""UA""","""United Air Lines Inc.""",1545,"""EWR""","""IAH"""
"""UA""","""United Air Lines Inc.""",1714,"""LGA""","""IAH"""
"""AA""","""American Airlines Inc.""",1141,"""JFK""","""MIA"""
"""B6""","""JetBlue Airways""",725,"""JFK""","""BQN"""
"""DL""","""Delta Air Lines Inc.""",461,"""LGA""","""ATL"""
…,…,…,…,…
"""B6""","""JetBlue Airways""",1806,"""JFK""","""BOS"""
"""UA""","""United Air Lines Inc.""",1187,"""EWR""","""LAS"""
"""B6""","""JetBlue Airways""",371,"""LGA""","""FLL"""


In [13]:
#question3.2
result = ctx.execute("""
SELECT
  f.carrier,
  AVG(2013 - p.year) AS avg_aircraft_age
FROM flights AS f
JOIN planes  AS p
  ON f.tailnum = p.tailnum
WHERE p.year IS NOT NULL
GROUP BY f.carrier
ORDER BY avg_aircraft_age DESC
""")
result

carrier,avg_aircraft_age
str,f64
"""MQ""",35.319
"""AA""",25.869426
"""DL""",16.372169
"""UA""",13.207691
"""FL""",11.385829
…,…
"""B6""",6.686702
"""F9""",4.87874
"""VX""",4.473643


In [14]:
#question3.3
_ = ctx.execute("""
    SELECT *
    FROM weather
    LIMIT 5
""")

result = ctx.execute("""
SELECT
  f.time_hour,
  f.origin,
  f.dest,
  f.flight,
  f.dep_delay,
  w.wind_speed,
  w.precip
FROM flights AS f
JOIN weather AS w
  ON f.origin = w.origin
 AND f.time_hour = w.time_hour
WHERE f.dep_delay > 30
  AND (w.wind_speed > 20 OR w.precip > 0.1)
ORDER BY f.dep_delay DESC
""")
result

time_hour,origin,dest,flight,dep_delay,wind_speed,precip
"datetime[μs, UTC]",str,str,i64,f64,f64,f64
2013-04-10 23:00:00 UTC,"""JFK""","""TPA""",2391,960.0,31.07106,0.11
2013-12-14 23:00:00 UTC,"""JFK""","""TPA""",2391,825.0,20.71404,0.01
2013-04-19 21:00:00 UTC,"""JFK""","""LAS""",257,797.0,25.31716,0.0
2013-04-19 21:00:00 UTC,"""JFK""","""IAH""",1901,761.0,25.31716,0.0
2013-04-10 23:00:00 UTC,"""LGA""","""MCO""",1485,639.0,33.37262,0.14
…,…,…,…,…,…,…
2013-08-02 00:00:00 UTC,"""LGA""","""ORD""",1128,31.0,13.80936,0.2
2013-08-07 22:00:00 UTC,"""LGA""","""ORD""",1424,31.0,20.71404,0.01
2013-08-13 14:00:00 UTC,"""EWR""","""ATL""",4140,31.0,9.20624,0.17


In [15]:
#question4
#question4.1
result = ctx.execute("""
SELECT
  p.manufacturer,
  p.model,
  COUNT(*) AS flights_cnt
FROM flights AS f
JOIN planes  AS p
  ON f.tailnum = p.tailnum
GROUP BY p.manufacturer, p.model
ORDER BY flights_cnt DESC
LIMIT 10
""")
result

manufacturer,model,flights_cnt
str,str,u32
"""AIRBUS""","""A320-232""",31278
"""EMBRAER""","""EMB-145LR""",28027
"""EMBRAER""","""ERJ 190-100 IGW""",23716
"""AIRBUS INDUSTRIE""","""A320-232""",14553
"""EMBRAER""","""EMB-145XR""",14051
"""BOEING""","""737-824""",13809
"""BOMBARDIER INC""","""CL-600-2D24""",11807
"""BOEING""","""737-7H4""",10389
"""BOEING""","""757-222""",9150


In [16]:
#question4.2
result = ctx.execute("""
SELECT
  f.origin,
  ao.name AS origin_name,
  f.dest,
  ad.name AS dest_name,
  COUNT(*) AS flights_cnt,
  AVG(f.dep_delay) AS avg_dep_delay,
  100.0 * AVG(CASE WHEN f.dep_delay > 30 THEN 1 ELSE 0 END) AS pct_delay_gt30
FROM flights AS f
JOIN airports AS ao
  ON f.origin = ao.faa
JOIN airports AS ad
  ON f.dest = ad.faa
GROUP BY f.origin, ao.name, f.dest, ad.name
ORDER BY flights_cnt DESC
LIMIT 10
""")
result

ColumnNotFoundError: name:ad

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
AGGREGATE
	[len().alias("flights_cnt"), col("dep_delay").mean().alias("avg_dep_delay"), [(100.0) * (when([(col("dep_delay")) > (30.0)]).then(dyn int: 1).otherwise(dyn int: 0).mean())].alias("pct_delay_gt30")] BY [col("origin"), col("name"), col("dest"), col("name:ad").alias("name")] FROM
  INNER JOIN:
  LEFT PLAN ON: [col("dest")]
    INNER JOIN:
    LEFT PLAN ON: [col("origin")]
      DF ["year", "month", "day", "dep_time", ...]; PROJECT */19 COLUMNS
    RIGHT PLAN ON: [col("faa")]
      DF ["faa", "name", "lat", "lon", ...]; PROJECT */8 COLUMNS
    END INNER JOIN
  RIGHT PLAN ON: [col("faa")]
    DF ["faa", "name", "lat", "lon", ...]; PROJECT */8 COLUMNS
  END INNER JOIN

In [ ]:
#Compare with Polars (Example 2.1)
sql_result = ctx.execute("""
    SELECT
        origin,
        AVG(dep_delay) AS avg_delay
    FROM flights
    WHERE dep_delay IS NOT NULL
    GROUP BY origin
    ORDER BY avg_delay DESC
""")

polars_result = (
    flights
    .filter(pl.col("dep_delay").is_not_null())
    .group_by("origin")
    .agg(pl.col("dep_delay").mean().alias("avg_delay"))
    .sort("avg_delay", descending=True)
)

print("SQL Result:")
print(sql_result)

print("\nPolars Result:")
print(polars_result)